## Descriptors
1. What descriptors are - the protocol
2. Data vs Non-data descriptors
3. How Python's attribute lookup works (mro, instance dict, class dict)
4. `__get__`, `__set__`, `__delete__`, `__set_name__`
5. Properties as descriptors (under the hood)
6. Real-world examples - validation, type checking, caching
7. Django applications
8. FastAPI applications
9. vs properties (the DRY argument)

---

### What Descriptors Actually Are

A descriptor is any object that defines at least one of `__get__`, `__set__`, `__delete__`. That's the entire protocol. Python calls these automatically during attribute access (with **dot** `.` - `<obj.attr>`) — **you never call them directly**.
```py
# This is ALL it takes to be a descriptor:
class MyDescriptor:
    def __get__(self, obj, objtype=None): ...   # triggered on attribute READ
    def __set__(self, obj, value): ...          # triggered on attribute WRITE
    def __delete__(self, obj): ...              # triggered on attribute DELETE
    def __set_name__(self, owner, name): ...    # triggered at CLASS CREATION TIME
```

---

### The Two Types — Critical Distinction
```shell
Data Descriptor        → defines __set__ (and/or __delete__)
                         OVERRIDES instance __dict__
                         Example: property, validated fields

Non-Data Descriptor    → defines ONLY __get__
                         instance __dict__ can shadow it
                         Example: functions/methods, cached_property
```

Python's attribute lookup order (MRO chain):

```shell
obj.attr  →
    1. Data descriptors on the CLASS (and its MRO)     ← highest priority
    2. Instance __dict__  (obj.__dict__["attr"])
    3. Non-data descriptors on the class + class vars  ← lowest priority
```
- This is why a property (data descriptor) always wins over instance dict
- This is why methods (non-data) can be shadowed by instance attributes

---

### `__set_name__` — The Game Changer
Without `__set_name__`, descriptors had to be told their own name. Python 3.6+ fixed this:

In [ ]:
class Descriptor:
    def __set_name__(self, owner, name):
        # owner → the CLASS that owns this descriptor  (e.g. User)
        # name  → the attribute name assigned to       (e.g. "username")
        self.name         = name
        self.storage_name = f"_{name}"  # where we store the actual value

class User:
    username = Descriptor()  # __set_name__(User, "username") called HERE at class creation, not instance!

**The Problem `__set_name__` Solves**

Without it, the descriptor **has no idea what name it was assigned to**:

In [ ]:
class Validated:
    def __set__(self, obj, value):
        if not isinstance(value, str):
            raise TypeError(f"??? must be a string")  # ← what do we put here?
            #               ↑ descriptor doesn't know its own name!

class User:
    name  = Validated()
    email = Validated()
    #  ↑                ↑
    #  the descriptor sitting here has no way to know
    #  it's called "name" or "email"

**The Two Parameters — Super Simply**

In [ ]:
def __set_name__(self, owner, name):
    pass

```shell
self  → the descriptor instance itself  (Validated())
owner → the CLASS that owns it          (User)
name  → the attribute name it's assigned to ("name" or "email")
```
**Python calls this automatically the moment you write:**

In [ ]:
class User:
    name = Validated()
    #  ↑
    #  Python sees this assignment and immediately calls:
    #  Validated().__set_name__(owner=User, name="name")

**Step-by-Step What Actually Happens**

In [ ]:
class Validated:
    def __set_name__(self, owner, name):
        print(f"  __set_name__ called!")
        print(f"  owner = {owner}")   # <class 'User'>
        print(f"  name  = {name!r}")  # 'name' or 'email'
        self.name = name              # NOW the descriptor knows its OWN name

class User:
    print("--- class body starting ---")
    name  = Validated()   # Python calls __set_name__(User, "name")
    email = Validated()   # Python calls __set_name__(User, "email")
    print("--- class body done ---")

```shell
Output:
--- class body starting ---
  __set_name__ called!
  owner = <class 'User'>
  name  = 'name'
  __set_name__ called!
  owner = <class 'User'>
  name  = 'email'
--- class body done ---
```
> `__set_name__` fires at class creation time, before any instance exists.

---

**Why Two Storage Names**

This is what confused you. There are **two separate things** to store:

In [ ]:
class Validated:
    def __set_name__(self, owner, name):
        self.name         = name          # "name" or "email"
        self.storage_name = f"_desc_{name}"  # "_desc_name" or "_desc_email"

```shell
self.name         → used in ERROR MESSAGES so you know which field failed
self.storage_name → used to store the ACTUAL VALUE on the instance

Why not use self.name to store the value too?
→ it would CONFLICT with the descriptor itself on the class!
```

In [ ]:
# ❌ If we stored value under the same name:
class Validated:
    def __set__(self, obj, value):
        setattr(obj, self.name, value)   # self.name = "name"
        # setattr(obj, "name", value)
        # This calls obj.name = value
        # Which triggers __set__ again
        # → infinite recursion! 💥

# ✅ We store under a private different name:
class Validated:
    def __set__(self, obj, value):
        setattr(obj, self.storage_name, value)  # self.storage_name = "_desc_name"
        # setattr(obj, "_desc_name", value)
        # Goes straight to instance __dict__ — no recursion ✅

**The Full Picture in One Example**

In [ ]:
class Validated:
    def __set_name__(self, owner, name):
        self.name         = name            # e.g. "email"
        self.storage_name = f"_desc_{name}" # e.g. "_desc_email"

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        return getattr(obj, self.storage_name)  # reads "_desc_email" from instance

    def __set__(self, obj, value):
        if not isinstance(value, str):
            raise TypeError(f"{self.name!r} must be a string")
            #                ↑ "email must be a string" — clear error message!
        setattr(obj, self.storage_name, value)  # stores in "_desc_email"


class User:
    name  = Validated()   # __set_name__(User, "name")  → self.name="name",  self.storage_name="_desc_name"
    email = Validated()   # __set_name__(User, "email") → self.name="email", self.storage_name="_desc_email"

    def __init__(self, name, email):
        self.name  = name    # → Validated.__set__ → stores in obj._desc_name
        self.email = email   # → Validated.__set__ → stores in obj._desc_email


u = User("Alice", "alice@example.com")

print(u.__dict__)
# {"_desc_name": "Alice", "_desc_email": "alice@example.com"}
#       ↑ never "name" or "email" directly — stored under private keys

print(u.name)    # → Validated.__get__ → reads u._desc_name  → "Alice"
print(u.email)   # → Validated.__get__ → reads u._desc_email → "alice@example.com"

u.email = 123    # → Validated.__set__ → TypeError: 'email' must be a string
#                                                     ↑ correct field name in error!

1. `__set_name__(self, owner, name)` is called once by Python
- when the class body is executed — not when an instance is created.

  - `owner` → tells the descriptor WHICH CLASS owns it  (rarely needed)
  - `name`  → tells the descriptor **ITS OWN ATTRIBUTE NAME**

2. We save `name` as `self.name` for **error messages**.
3. We save `f"_desc_{name}"` as `self.storage_name` to **avoid infinite recursion** when reading/writing the actual value on the instance.

4. Without `__set_name__`, the descriptor is **blind** — it doesn't know if it was assigned to "email" or "price" or anything else.

---

### Property Is Just a Descriptor — Under the Hood

In [ ]:
# What @property actually is:
class property:
    def __get__(self, obj, objtype=None):
        if obj is None:
            return self        # accessed on class → return descriptor itself
        return self.fget(obj)  # accessed on instance → call the getter

    def __set__(self, obj, value):
        if self.fset is None:
            raise AttributeError("can't set attribute")
        self.fset(obj, value)

    def __delete__(self, obj):
        if self.fdel is None:
            raise AttributeError("can't delete attribute")
        self.fdel(obj)

In [ ]:
# These are IDENTICAL:

# ── Using @property ──────────────────────────────────────────
class Circle:
    def __init__(self, radius):
        self._radius = radius

    @property
    def radius(self):
        return self._radius

    @radius.setter
    def radius(self, value):
        if value < 0:
            raise ValueError("Radius cannot be negative")
        self._radius = value


# ── What Python actually builds: ─────────────────────────────
class Circle:
    def __init__(self, radius):
        self._radius = radius

    def _get_radius(self):
        return self._radius

    def _set_radius(self, value):
        if value < 0:
            raise ValueError("Radius cannot be negative")
        self._radius = value

    radius = property(_get_radius, _set_radius)  # property(...) is a descriptor instance sitting on the class

In [4]:
cir = Circle(5)
cir.radius = 6

### The DRY Problem With Properties

In [ ]:
# ❌ Classic property anti-pattern — you've seen this:
class Product:
    @property
    def name(self):
        return self._name

    @name.setter
    def name(self, value):
        if not isinstance(value, str):
            raise TypeError("name must be a string")
        if not value.strip():
            raise ValueError("name cannot be empty")
        self._name = value.strip()

    @property
    def brand(self):
        return self._brand

    @brand.setter
    def brand(self, value):
        if not isinstance(value, str):       # ← identical logic
            raise TypeError("brand must be a string")
        if not value.strip():                # ← identical logic
            raise ValueError("brand cannot be empty")
        self._brand = value.strip()          # ← identical logic

    @property
    def category(self):
        return self._category

    @category.setter
    def category(self, value):
        if not isinstance(value, str):       # ← identical logic AGAIN
            raise TypeError("category must be a string")
        if not value.strip():                # ← identical logic AGAIN
            raise ValueError("category cannot be empty")
        self._category = value.strip()       # ← identical logic AGAIN

# 3 fields, 30 lines, zero reuse. Add 5 more fields → 80 lines of duplication.

### The Descriptor Solution — Write Once, Apply Everywhere

In [1]:
# descriptors.py

class ValidatedString:
    """
    Data descriptor — reusable string validation.
    Replaces 10 lines of @property with 1 line of class attribute.
    """
    def __set_name__(self, owner, name):
        self.name         = name
        self.storage_name = f"_desc_{name}"   # avoids collision with user attrs

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self   # Product.name → returns descriptor, not value
        return getattr(obj, self.storage_name, None)

    def __set__(self, obj, value):
        if not isinstance(value, str):
            raise TypeError(f"{self.name!r} must be a string, got {type(value).__name__}")
        value = value.strip()
        if not value:
            raise ValueError(f"{self.name!r} cannot be empty or whitespace")
        setattr(obj, self.storage_name, value)

    def __delete__(self, obj):
        raise AttributeError(f"{self.name!r} cannot be deleted")

In [2]:
# ✅ Clean model using the descriptor:
class Product:
    name     = ValidatedString()
    brand    = ValidatedString()
    category = ValidatedString()

    def __init__(self, name: str, brand: str, category: str):
        self.name     = name      # triggers ValidatedString.__set__
        self.brand    = brand     # triggers ValidatedString.__set__
        self.category = category  # triggers ValidatedString.__set__


# Usage:
p1 = Product("MacBook Pro", "Apple", "Laptops")
print(p1.name)    # "MacBook Pro"  → triggers __get__

# p1.name = "  "    # raises ValueError: 'name' cannot be empty
# p1.name = 123     # raises TypeError: 'name' must be a string
# del p1.name       # raises AttributeError: 'name' cannot be deleted

# # Where is the value stored?
# print(p1.__dict__)
# # {'_desc_name': 'MacBook Pro', '_desc_brand': 'Apple', '_desc_category': 'Laptops'}
# #         ↑ descriptor stores in private key, never conflicts with descriptor itself
# p2 = Product("IPhone", "Apple", "Smart Phones")
# print(p2.name)

MacBook Pro


### Complete Mental Model
```shell
Descriptor Protocol:

__get__(self, obj, objtype)   READ   obj.attr  / Class.attr
__set__(self, obj, value)     WRITE  obj.attr = value
__delete__(self, obj)         DEL    del obj.attr
__set_name__(owner, name)     HOOK   called once at class body execution


Data vs Non-Data:

has __set__?    YES → Data descriptor    → beats instance __dict__
                NO  → Non-data descriptor → instance __dict__ wins


Attribute lookup (obj.attr):

  1. type(obj).__mro__ → search for Data descriptor    ← wins if found
  2. obj.__dict__[attr]                                ← wins if no data desc
  3. type(obj).__mro__ → search for Non-data descriptor


Where to use descriptors in real projects:

  Properties (1 field)       → @property is fine
  Same validation (2+ fields) → descriptor pays off immediately
  Cross-model reuse           → descriptor in shared descriptors.py
  Caching (non-data)          → CachedProperty / functools.cached_property
  Audit/tracking              → Audited descriptor on model fields
  Django models               → Python-level validation layer above ORM
  FastAPI services            → request-scoped lazy cache
```

---

> **The interview answer**: "Properties are descriptors under the hood — they're perfect for a single field but break DRY the moment the same validation appears on two fields. A custom descriptor writes the logic once and deploys it as a single class attribute. I reach for descriptors specifically when I see the same getter/setter pattern repeated across fields or across models. In Django I use them for Python-level field validation on top of the ORM, and in FastAPI service classes I use non-data descriptors for request-scoped lazy caching — which is exactly how functools.cached_property works internally."